In [1]:
import pandas as pd

train = pd.read_csv('household_train.csv')
valid = pd.read_csv('household_validation.csv')
test = pd.read_csv('household_test.csv')

In [2]:
input_width = 60
label_width = 30
feature_columns = ['Global_active_power']

In [3]:
import numpy as np
import tensorflow as tf

def create_sequences(data, input_width, label_width, feature_columns, target_column="Global_active_power"):
    X, y = [], []
    values_X = data[feature_columns].values
    values_y = data[target_column].values
    for i in range(len(data) - input_width - label_width):
        X.append(values_X[i:i+input_width])
        y.append(values_y[i+input_width:i+input_width+label_width])
    return np.array(X)[..., np.newaxis], np.array(y)

X_train, y_train = create_sequences(train, input_width, label_width, ["Global_active_power"])
X_valid, y_valid = create_sequences(valid, input_width, label_width, ["Global_active_power"])
X_test,  y_test  = create_sequences(test,  input_width, label_width, ["Global_active_power"])


2025-09-07 11:46:00.847353: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Tuning

In [4]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
import random

def build_model(input_width, n_features, label_width,
                filters, kernel_size, dense_units, dropout_rate, learning_rate,
                use_pooling=True):
    model = Sequential()
    model.add(Conv1D(filters=filters, kernel_size=kernel_size,
                     activation='relu',
                     input_shape=(input_width, n_features)))
    
    if use_pooling:
        model.add(MaxPooling1D(pool_size=2))
    
    model.add(Flatten())
    model.add(Dense(dense_units, activation='relu'))
    model.add(Dropout(dropout_rate))
    model.add(Dense(label_width))
    
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  loss="mse", metrics=["mae"])
    return model


In [5]:
# Suchraum definieren
param_grid = {
    "filters": [16, 32, 64],
    "kernel_size": [2, 3, 5],
    "dense_units": [32, 64, 128],
    "dropout_rate": [0.0, 0.2, 0.5],
    "learning_rate": [0.001, 0.0005],
    "batch_size": [32, 64],
    "use_pooling": [True, False]
}

n_trials = 70  # z. B. 10 zufällige Kombinationen

results = []

for i in range(n_trials):
    params = {k: random.choice(v) for k, v in param_grid.items()}
    print(f"Trial {i+1}: {params}")

    # batch_size herausziehen
    batch_size = params.pop("batch_size")

    # Modell bauen (ohne batch_size)
    model = build_model(input_width, len(feature_columns), label_width, **params)

    # Trainieren (mit batch_size)
    history = model.fit(
        X_train, y_train,
        validation_data=(X_valid, y_valid),
        epochs=2,
        batch_size=batch_size,
        verbose=0
    )
    val_mae = min(history.history["val_mae"])
    results.append((params | {"batch_size": batch_size}, val_mae))


# Beste Parameter auswählen
best_params, best_score = sorted(results, key=lambda x: x[1])[0]
print("Beste Kombination:", best_params, "mit val_mae:", best_score)


Trial 1: {'filters': 32, 'kernel_size': 5, 'dense_units': 128, 'dropout_rate': 0.2, 'learning_rate': 0.0005, 'batch_size': 32, 'use_pooling': False}


/Users/basti/miniforge3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Trial 2: {'filters': 32, 'kernel_size': 5, 'dense_units': 64, 'dropout_rate': 0.2, 'learning_rate': 0.0005, 'batch_size': 64, 'use_pooling': True}
Trial 3: {'filters': 32, 'kernel_size': 3, 'dense_units': 128, 'dropout_rate': 0.5, 'learning_rate': 0.0005, 'batch_size': 64, 'use_pooling': True}
Trial 4: {'filters': 32, 'kernel_size': 5, 'dense_units': 128, 'dropout_rate': 0.0, 'learning_rate': 0.001, 'batch_size': 64, 'use_pooling': True}
Trial 5: {'filters': 64, 'kernel_size': 2, 'dense_units': 64, 'dropout_rate': 0.2, 'learning_rate': 0.0005, 'batch_size': 32, 'use_pooling': False}
Trial 6: {'filters': 16, 'kernel_size': 5, 'dense_units': 64, 'dropout_rate': 0.2, 'learning_rate': 0.0005, 'batch_size': 64, 'use_pooling': False}
Trial 7: {'filters': 32, 'kernel_size': 3, 'dense_units': 64, 'dropout_rate': 0.0, 'learning_rate': 0.0005, 'batch_size': 32, 'use_pooling': True}
Trial 8: {'filters': 64, 'kernel_size': 5, 'dense_units': 128, 'dropout_rate': 0.5, 'learning_rate': 0.001, 'batch_

In [6]:
# %%
# Fixe Konfiguration
fixed_params = {
    "filters": 32,
    "kernel_size": 2,
    "dense_units": 128,
    "dropout_rate": 0.0,
    "learning_rate": 0.0005,
    "use_pooling": False
}
batch_size = 32

# Modell bauen
model = build_model(input_width, len(feature_columns), label_width, **fixed_params)

# Trainieren
history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=2,   # kannst du je nach Bedarf ändern
    batch_size=batch_size,
    verbose=1
)

# Bestes val_mae extrahieren
val_mae = min(history.history["val_mae"])
print("Validation MAE für fixe Konfiguration:", val_mae)


The history saving thread hit an unexpected error (OperationalError('unable to open database file')).History will not be written to the database.
Epoch 1/2
50005/50005 ━━━━━━━━━━━━━━━━━━━━ 205s 4ms/step - loss: 0.4555 - mae: 0.3793 - val_loss: 0.3985 - val_mae: 0.3320
Epoch 2/2
50005/50005 ━━━━━━━━━━━━━━━━━━━━ 224s 4ms/step - loss: 0.4399 - mae: 0.3695 - val_loss: 0.3902 - val_mae: 0.3424
Validation MAE für fixe Konfiguration: 0.3319683372974396
